# Heat Pump Workflow

Compare a base case against a candidate heat-pump integration scenario.

This notebook is about targeting and integration. The main question is not
whether a cycle can be solved in isolation. The main question is whether a
useful temperature lift displaces the right external utilities.


In [ ]:
import json
from copy import deepcopy
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

from OpenPinch import PinchProblem
from OpenPinch.resources import copy_sample_case

workspace = TemporaryDirectory()
work_dir = Path(workspace.name)
base_case_path = copy_sample_case("basic_pinch.json", work_dir / "basic_pinch.json")
base_problem = PinchProblem(problem_filepath=base_case_path)
base_summary = base_problem.summary_frame()
base_summary.loc[base_summary["Target"] == "Plant/Direct Integration"]


## Candidate integration scenario

The scenario below inserts a small condenser and evaporator pair into the plant
case. The condenser rejects 500 kW at about 170 degC and the evaporator lifts
400 kW from about 90 degC. The 100 kW difference is a simple stand-in for the
compressor-power requirement.

The point here is to test the integration effect first: how do the utility
numbers and the grand composite curve change after introducing that lift?


In [ ]:
base_data = json.loads(base_case_path.read_text(encoding="utf-8"))
integrated_data = deepcopy(base_data)

hp_streams = [
    {
        "zone": "Plant",
        "name": "HP Condenser",
        "t_supply": {"value": 170.0, "units": "degC"},
        "t_target": {"value": 169.9, "units": "degC"},
        "heat_flow": {"value": 500.0, "units": "kW"},
        "dt_cont": {"value": 0.0, "units": "degC"},
        "htc": {"value": 1.0, "units": "kW/m^2/degC"},
    },
    {
        "zone": "Plant",
        "name": "HP Evaporator",
        "t_supply": {"value": 89.9, "units": "degC"},
        "t_target": {"value": 90.0, "units": "degC"},
        "heat_flow": {"value": 400.0, "units": "kW"},
        "dt_cont": {"value": 0.0, "units": "degC"},
        "htc": {"value": 1.0, "units": "kW/m^2/degC"},
    },
]

integrated_data["streams"].extend(hp_streams)
integrated_case_path = work_dir / "basic_pinch_with_hp.json"
integrated_case_path.write_text(json.dumps(integrated_data), encoding="utf-8")

integrated_problem = PinchProblem(problem_filepath=integrated_case_path)
integrated_summary = integrated_problem.summary_frame()
integrated_summary.loc[integrated_summary["Target"] == "Plant/Direct Integration"]


In [ ]:
def _plant_row(frame, label):
    row = frame.loc[frame["Target"] == "Plant/Direct Integration"].iloc[0]
    return pd.Series(
        {
            "Hot Utility Target": row["Hot Utility Target"],
            "Cold Utility Target": row["Cold Utility Target"],
            "Heat Recovery": row["Heat Recovery"],
            "Cold Pinch": row["Cold Pinch"],
        },
        name=label,
    )

comparison = pd.DataFrame(
    [
        _plant_row(base_summary, "Base case"),
        _plant_row(integrated_summary, "Integrated heat-pump scenario"),
    ]
)
comparison.loc["Change"] = (
    comparison.loc["Integrated heat-pump scenario"] - comparison.loc["Base case"]
)
comparison["Approx. HP power input"] = [None, 100.0, None]
comparison


## Reading the result

For this scenario, the plant hot utility target drops from about 750 kW to
500 kW and the cold utility target drops from about 1000 kW to 850 kW. That is
the main targeting result. The cycle-power estimate matters, but it is not the
first screening question.

If a candidate heat-pump integration lowers hot utility demand but pushes cold
utility up too far, or requires an unrealistic lift, it is probably not the
right placement.


In [ ]:
base_gcc = base_problem.plot_grand_composite_curve(zone_name="Plant/Direct Integration")
integrated_gcc = integrated_problem.plot_grand_composite_curve(
    zone_name="Plant/Direct Integration"
)
[
    base_gcc.layout.title.text,
    integrated_gcc.layout.title.text,
    len(base_gcc.data),
    len(integrated_gcc.data),
]
